In [0]:
## Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

##Tests for Customer Table


In [0]:
##Data Ingestion for Customers table

customers=spark.table("shop_performance.shop.customers").toPandas()
display(customers)

In [0]:
##Checking the information of the table
customers.info()

In [0]:
## Printing the Column list for this dataset
print(customers.columns.tolist())

In [0]:
##Checking for duplicates 
customers.duplicated().sum()

In [0]:
##Checking for Unique values
print(customers["CustomerID"].value_counts(dropna=False))

In [0]:
##Checking for null values
print(customers.isnull().sum())

In [0]:
##Checking for minimum, maximum, and average
age_numeric = pd.to_numeric(customers["Age"], errors="coerce")
print(age_numeric.min())
print(age_numeric.max())
print(age_numeric.mean())

In [0]:
## Replacing the Null Age values with Average age
customers['Age'] = pd.to_numeric(customers['Age'], errors='coerce')
mean_age = round(customers['Age'].mean(),0)
customers['Age'] = customers['Age'].fillna(mean_age)


In [0]:
##Replacing the Null Cities with "Unknown"
customers["City"] = customers["City"].fillna("Unknown")

In [0]:
##Changing the names of the City to One name
customers["City"] = customers["City"].str.title()


In [0]:
##Changing the name of the City
customers["City"] = customers["City"].str.strip().str.title()
customers["City"] = customers["City"].replace("Mashad", "Mashhad")

In [0]:
## Ensuring that the dates are well structured and removing Null dates with modal date
customers["SignupDate"] = pd.to_datetime(customers["SignupDate"], errors="coerce")
customers["SignupDate"] = customers["SignupDate"].fillna(customers["SignupDate"].mode())


In [0]:
##Replacing the Null CustomerSegment with "Unknown"
customers["CustomerSegment"] = customers["CustomerSegment"].fillna("Unknown")

In [0]:
print(customers.describe(include="all"))

##Tests for Order Table

In [0]:
##Data Ingestion for Orders table

orders=spark.table("shop_performance.shop.orders")
orders=orders.toPandas()
display(orders)

In [0]:
##Check the orders table information
orders.info()

In [0]:
##Checking for duplicates 
orders.duplicated().sum()

In [0]:
## Removing Duplicates on the Orders Table
orders = orders.drop_duplicates()

In [0]:
print(orders.describe(include="all"))

In [0]:
## Printing the Column list for this dataset
print(orders.columns.tolist())

In [0]:
## Unique customers
print("Unique Customers:", orders["CustomerID"].nunique())

## Unique products
print("Unique Products:", orders["ProductID"].nunique())

In [0]:
##Checking the Min and Max of the data points Quantity and Discount values
orders["Quantity"] = pd.to_numeric(orders["Quantity"], errors="coerce")
orders["Discount"] = pd.to_numeric(orders["Discount"], errors="coerce")
print(orders["Quantity"].min(), orders["Quantity"].max())
print(orders["Discount"].min(), orders["Discount"].max())


In [0]:
print(orders.isnull().sum())

In [0]:
##Repalcing the Blank spaces with a mean value in the Discount column and Ensuring there are no errors
orders["Discount"] = pd.to_numeric(orders["Discount"], errors="coerce")
orders["Discount"] = orders["Discount"].fillna(orders["Discount"].mean())

In [0]:
##Repalcing the Blank spaces with a mean value in the Quantity column and Ensuring there are no errors
orders["Quantity"] = pd.to_numeric(orders["Quantity"], errors="coerce")
orders["Quantity"] = orders["Quantity"].fillna(orders["Quantity"].mean())

In [0]:
##Replacing Null payment methods with Unknown
orders["PaymentMethod"] = orders["PaymentMethod"].fillna("Unknown")

In [0]:
## Ensuring there dates are well structured and replaced Null dates with Modal dates
orders["OrderDate"] = pd.to_datetime(orders["OrderDate"], errors="coerce")
orders["OrderDate"] = orders["OrderDate"].fillna(orders["OrderDate"].mode())

In [0]:
print(orders.describe(include="all"))

##----------------------------------------------------------------------------------------------------------

##Tests For Payments Table

In [0]:
##Data Ingestion for Payments Table

payments=spark.table("shop_performance.shop.payments")
payments=payments.toPandas()
display(payments)

In [0]:
payments.info()

In [0]:
##Checking for payments duplicates
payments.duplicated().sum()

In [0]:
##Checking for Null values 
print(payments.isnull().sum())

In [0]:
## Ensuring there dates are well structured and replaced Null dates with Modal dates
payments["PaymentDate"] = pd.to_datetime(payments["PaymentDate"], errors="coerce")
payments["PaymentDate"] = payments["PaymentDate"].fillna(payments["PaymentDate"].mode())

In [0]:
##Reviewing the statistics of the data
print(payments.describe(include="all"))

##Tests For Products Table

In [0]:
##Data Ingestion for Products Table

products=spark.table("shop_performance.shop.products")
products=products.toPandas()
display(products)

In [0]:
products.info()

In [0]:
##Checking for products duplicates
products.duplicated().sum()

In [0]:
##Reviewing the Null Values on Products Table
print(products.isnull().sum())

In [0]:
##Correcting the Numeric Errors and using Mean to Replace Nulls even though they are not there
products["UnitPrice"] = pd.to_numeric(products["UnitPrice"], errors="coerce")
products["UnitPrice"] = products["UnitPrice"].fillna(products["UnitPrice"].mean())

In [0]:
##Chekcing the Min and Max 
print(products["UnitPrice"].min(), products["UnitPrice"].max())

In [0]:
##Reviewing the statistics of the data
print(products.describe(include="all"))

##----------------------------------------------------

##Consolidated Code from Customers, Orders, Payments and Products

In [0]:
data = orders.merge(
    products,
    on="ProductID",
    how="left"
)

data = data.merge(
    customers,
    on="CustomerID",
    how="left"
)

payment_status = payments.groupby("OrderID")["PaymentStatus"].first().reset_index()

data = data.merge(
    payment_status,
    on="OrderID",
    how="left"
)
display(data)

In [0]:
##Calculate Revenue using the Quantity, Unitprice and the Discount 
data["Revenue"] = (
    data["Quantity"]
    * data["UnitPrice"]
    * (1 - data["Discount"])
)

In [0]:
##Excluding the revenue on orders that were returned or canceled by customers
revenue_data = data[
    ~data["Status"].str.lower().isin(["cancelled", "canceled", "returned"])
].copy()

In [0]:
##Conducting date function accroding to day name, month name, and year
revenue_data["OrderDate"] = pd.to_datetime(revenue_data["OrderDate"])
revenue_data["Year"] = revenue_data["OrderDate"].dt.year
revenue_data["Month"] = revenue_data["OrderDate"].dt.month
revenue_data["YearMonth"] = revenue_data["OrderDate"].dt.to_period("M")

In [0]:
total_revenue = revenue_data["Revenue"].sum()

total_orders = revenue_data["OrderID"].nunique()

average_order_value = total_revenue / total_orders

In [0]:
monthly_revenue = (
    revenue_data
    .groupby("YearMonth")["Revenue"]
    .sum()
    .reset_index()
)

In [0]:
##Monthly Revenue growth
monthly_revenue["Growth"] = (
    monthly_revenue["Revenue"].pct_change() * 100
)

In [0]:
## Ensuring that the dates are well structured and removing Null dates with modal date
data["SignupDate"] = pd.to_datetime(data["SignupDate"], errors="coerce")
data["SignupDate"] = data["SignupDate"].fillna(data["SignupDate"].mode())

In [0]:
## Ensuring that the dates are well structured and removing Null dates with modal date
data["OrderDate"] = pd.to_datetime(data["OrderDate"], errors="coerce")
data["OrderDate"] = data["OrderDate"].fillna(data["OrderDate"].mode())

In [0]:
## Replacing the Null Age values with Average age
data['Age'] = pd.to_numeric(data['Age'], errors='coerce')
mean_age = round(data['Age'].mean(),0)
data['Age'] = data['Age'].fillna(mean_age)

In [0]:
##Replacing the Null CustomerSegment with "Unknown"
data["CustomerSegment"] = data["CustomerSegment"].fillna("Unknown")

In [0]:
##Replacing the Null Cities with "Unknown"
data["City"] = data["City"].fillna("Unknown")

In [0]:
##Conducting the date Functions
data["SignupDate"] = pd.to_datetime(data["SignupDate"])

data["Month"] = data["SignupDate"].dt.month_name()
data["Day"] = data["SignupDate"].dt.day_name()
data["Year"] = data["SignupDate"].dt.year

In [0]:
##Conducting the IF statement to categorize age groups
age_numeric = pd.to_numeric(customers["Age"], errors="coerce")
def age_group(age):
    if age <= 19:
        return "Teenagers"
    elif age <= 35:
        return "Youth"
    elif age <= 64:
        return "Adults"
    else:
        return "Pensioners"

data["Age_Group"] = pd.to_numeric(data["Age"], errors="coerce").apply(age_group)

In [0]:
##Categorizing the UnitPrices
def price_category(price):
    if price < 35:
        return "Cheaper"
    elif price < 65:
        return "Cheap"
    elif price <= 120:
        return "Pricey"
    else:
        return "Expensive"

data["Price_Category"] = data["UnitPrice"].apply(price_category)

In [0]:
##Categorizing the Quantity 
def quantity_category(quantity):
    if quantity <= 1:
        return "Low Quantity"
    elif quantity <= 3:
        return "Medium Quantity"
    elif quantity <= 5:
        return "High Quantity"
    else:
        return "Too Much"

data["Quantity_Category"] = data["Quantity"].apply(quantity_category)

In [0]:
##Categorizing the Revenue by Worst to great
def revenue_category(revenue):
    if revenue <= 0:
        return "Worst Revenue"
    elif revenue <= 100:
        return "Bad Revenue"
    elif revenue <= 200:
        return "Good Revenue"
    elif revenue <= 500:
        return "Great Revenue"
    elif revenue <= 1300:
        return "Best Revenue"
    else:
        return "Above"
        
data["Revenue_Category"] = data["Revenue"].apply(revenue_category)


In [0]:
##Displaying the Final Data
display(data)

In [0]:
data.info()